# Módulo I5 · Deep learning

**Caso:** una empresa de logística quiere leer automáticamente los códigos postales escritos a mano en los paquetes. Empezamos con un problema reducido: reconocer dígitos de 8×8 píxeles.

Objetivos:
1. Entender las piezas de una red neuronal: capas, activaciones, pérdida, optimizador.
2. Escribir un bucle de entrenamiento en **PyTorch**.
3. Comparar un perceptrón multicapa (MLP) con una red convolucional (CNN) y con un modelo clásico.
4. Detectar sobreajuste con curvas de aprendizaje y combatirlo con *dropout* y *early stopping*.

> Este notebook funciona en CPU en pocos minutos. En Colab puedes activar GPU en *Entorno de ejecución → Cambiar tipo*.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix

sns.set_theme(style="whitegrid")
torch.manual_seed(0)
dispositivo = "cuda" if torch.cuda.is_available() else "cpu"
print("Usando:", dispositivo)

## 1. Datos

In [ ]:
digits = load_digits()
X = (digits.images / 16.0).astype(np.float32)   # píxeles en [0, 1]
y = digits.target
X_tmp, X_test, y_tmp, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=0)
X_train, X_val, y_train, y_val = train_test_split(X_tmp, y_tmp, test_size=0.2, stratify=y_tmp, random_state=0)
print(f"train: {len(X_train)} | validación: {len(X_val)} | test: {len(X_test)}")

def loader(X, y, batch=64, shuffle=False):
    return DataLoader(TensorDataset(torch.tensor(X).unsqueeze(1), torch.tensor(y)), batch_size=batch, shuffle=shuffle)

dl_train, dl_val, dl_test = loader(X_train, y_train, shuffle=True), loader(X_val, y_val), loader(X_test, y_test)

## 2. Referencia clásica
Antes de usar deep learning, mide lo que consigue un modelo sencillo.

In [ ]:
base = LogisticRegression(max_iter=2000).fit(X_train.reshape(len(X_train), -1), y_train)
acc_base = base.score(X_test.reshape(len(X_test), -1), y_test)
print(f"Regresión logística – exactitud test: {acc_base:.3f}")

## 3. Las piezas de una red
- **Capa lineal**: $z = Wx + b$
- **Activación** no lineal (ReLU: $\max(0, z)$): sin ella, apilar capas equivale a una sola capa lineal.
- **Pérdida** (entropía cruzada): mide lo lejos que están las probabilidades predichas de la clase real.
- **Optimizador** (Adam): ajusta $W$ y $b$ en la dirección que reduce la pérdida, usando gradientes calculados por **retropropagación**.

In [ ]:
z = torch.linspace(-3, 3, 100)
fig, axes = plt.subplots(1, 3, figsize=(12, 2.8))
for ax, (nombre, f) in zip(axes, [("ReLU", torch.relu), ("Sigmoide", torch.sigmoid), ("Tanh", torch.tanh)]):
    ax.plot(z, f(z)); ax.set_title(nombre)
plt.tight_layout(); plt.show()

## 4. Bucle de entrenamiento reutilizable

In [ ]:
def exactitud(modelo, dl):
    modelo.eval()
    aciertos = 0
    with torch.no_grad():
        for xb, yb in dl:
            aciertos += (modelo(xb.to(dispositivo)).argmax(1).cpu() == yb).sum().item()
    return aciertos / len(dl.dataset)

def entrenar(modelo, epocas=40, lr=1e-3, paciencia=None):
    modelo.to(dispositivo)
    opt = torch.optim.Adam(modelo.parameters(), lr=lr)
    perdida_fn = nn.CrossEntropyLoss()
    hist = {"perdida_train": [], "acc_train": [], "acc_val": []}
    mejor, mejor_estado, sin_mejora = 0, None, 0
    for _ in range(epocas):
        modelo.train()
        total = 0.0
        for xb, yb in dl_train:
            xb, yb = xb.to(dispositivo), yb.to(dispositivo)
            opt.zero_grad()
            perdida = perdida_fn(modelo(xb), yb)
            perdida.backward()      # retropropagación
            opt.step()              # actualización de pesos
            total += perdida.item() * len(xb)
        hist["perdida_train"].append(total / len(dl_train.dataset))
        hist["acc_train"].append(exactitud(modelo, dl_train))
        hist["acc_val"].append(exactitud(modelo, dl_val))
        if paciencia is not None:   # early stopping
            if hist["acc_val"][-1] > mejor:
                mejor, sin_mejora = hist["acc_val"][-1], 0
                mejor_estado = {k: v.clone() for k, v in modelo.state_dict().items()}
            else:
                sin_mejora += 1
                if sin_mejora >= paciencia:
                    break
    if mejor_estado is not None:
        modelo.load_state_dict(mejor_estado)
    return hist

def curvas(hist, titulo):
    fig, axes = plt.subplots(1, 2, figsize=(11, 3))
    axes[0].plot(hist["perdida_train"]); axes[0].set(title="Pérdida (train)", xlabel="época")
    axes[1].plot(hist["acc_train"], label="train"); axes[1].plot(hist["acc_val"], label="validación")
    axes[1].set(title="Exactitud", xlabel="época"); axes[1].legend()
    fig.suptitle(titulo); plt.tight_layout(); plt.show()

## 5. Perceptrón multicapa (MLP)
Aplana la imagen a un vector de 64 valores: **ignora** qué píxeles son vecinos.

In [ ]:
mlp = nn.Sequential(
    nn.Flatten(),
    nn.Linear(64, 128), nn.ReLU(),
    nn.Linear(128, 64), nn.ReLU(),
    nn.Linear(64, 10),
)
print("Parámetros MLP:", sum(p.numel() for p in mlp.parameters()))
hist_mlp = entrenar(mlp)
curvas(hist_mlp, "MLP")
print(f"MLP – exactitud test: {exactitud(mlp, dl_test):.3f}")

## 6. Red convolucional (CNN)
Los filtros convolucionales recorren la imagen buscando patrones locales (bordes, curvas) **en cualquier posición**. Comparten pesos, por lo que necesitan menos parámetros y generalizan mejor en imágenes.

In [ ]:
cnn = nn.Sequential(
    nn.Conv2d(1, 16, kernel_size=3, padding=1), nn.ReLU(),   # 16 × 8 × 8
    nn.MaxPool2d(2),                                          # 16 × 4 × 4
    nn.Conv2d(16, 32, kernel_size=3, padding=1), nn.ReLU(),  # 32 × 4 × 4
    nn.MaxPool2d(2),                                          # 32 × 2 × 2
    nn.Flatten(),
    nn.Dropout(0.3),
    nn.Linear(32 * 2 * 2, 10),
)
print("Parámetros CNN:", sum(p.numel() for p in cnn.parameters()))
hist_cnn = entrenar(cnn, epocas=80, paciencia=10)
curvas(hist_cnn, "CNN con dropout y early stopping")
print(f"CNN – exactitud test: {exactitud(cnn, dl_test):.3f}")

## 7. ¿Qué ha aprendido la CNN?
Los filtros de la primera capa son pequeños detectores de patrones:

In [ ]:
filtros = cnn[0].weight.detach().cpu().squeeze(1)
fig, axes = plt.subplots(2, 8, figsize=(10, 2.8))
for ax, f in zip(axes.flat, filtros):
    ax.imshow(f, cmap="RdBu"); ax.axis("off")
plt.suptitle("Filtros 3×3 de la primera capa"); plt.show()

## 8. Análisis de errores

In [ ]:
cnn.eval()
with torch.no_grad():
    pred = cnn(torch.tensor(X_test).unsqueeze(1).to(dispositivo)).argmax(1).cpu().numpy()
sns.heatmap(confusion_matrix(y_test, pred), annot=True, fmt="d", cmap="Blues", cbar=False)
plt.xlabel("predicho"); plt.ylabel("real"); plt.show()

errores = np.where(pred != y_test)[0][:8]
fig, axes = plt.subplots(1, max(len(errores), 1), figsize=(12, 1.8))
for ax, i in zip(np.atleast_1d(axes), errores):
    ax.imshow(X_test[i], cmap="gray_r"); ax.set_title(f"real {y_test[i]} / pred {pred[i]}", fontsize=8); ax.axis("off")
plt.show()

## 9. Resumen

In [ ]:
print(f"{'Regresión logística':22s} {acc_base:.3f}")
print(f"{'MLP':22s} {exactitud(mlp, dl_test):.3f}")
print(f"{'CNN':22s} {exactitud(cnn, dl_test):.3f}")

Con imágenes tan pequeñas y limpias, el modelo clásico ya es muy bueno. La ventaja del deep learning crece con datos más grandes y complejos (fotos reales, texto, audio), donde además se suele partir de modelos **preentrenados** (*transfer learning*).

## Ejercicios

1. Entrena el MLP sin capas ocultas (`nn.Flatten(), nn.Linear(64, 10)`). ¿A qué modelo clásico equivale?
2. Duplica el número de neuronas del MLP y entrena 150 épocas sin early stopping. ¿Ves sobreajuste en las curvas?
3. Prueba tasas de aprendizaje de 1e-4, 1e-3 y 1e-2. ¿Qué ocurre con la curva de pérdida en cada caso?
4. **Aumento de datos:** desplaza aleatoriamente cada imagen de entrenamiento ±1 píxel (`np.roll`) y reentrena la CNN. ¿Mejora la exactitud de test?

In [ ]:
# TU CÓDIGO AQUÍ